In [0]:
!pip install databricks-langchain

In [0]:
dbutils.library.restartPython()

# Step 1: Add Logging Setup

In [0]:
import logging
import time
import json
from typing import TypedDict
from langgraph.graph import StateGraph, END
from databricks_langchain import ChatDatabricks
from langchain_core.messages import HumanMessage

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

# Step 2: Initialize Model (With Logging)

In [0]:
logger.info("Initializing Databricks Chat Model...")

chat_model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b",
    temperature=0.1,
    max_tokens=512
)

logger.info("Model initialized successfully.")

# Step 3: Define Graph State

In [0]:
class GraphState(TypedDict):
    topic: str
    explanation: str
    summary: str

# Step 4: Add Deep Debug Nodes

## Node 1: Generate Explanation

In [0]:
def generate_explanation(state: GraphState):
    logger.info("----- ENTERING NODE: generate_explanation -----")
    logger.info(f"Incoming State: {json.dumps(state, indent=2)}")

    start_time = time.time()

    prompt = f"Explain {state['topic']} in detail."
    logger.info(f"Prompt Sent To Model:\n{prompt}")

    response = chat_model.invoke(prompt)

    end_time = time.time()

    logger.info(f"Raw Response Object: {response}")
    logger.info(f"Response Content:\n{response.content}")

    # Token usage (if available)
    if hasattr(response, "response_metadata"):
        logger.info(f"Token Usage Metadata: {response.response_metadata}")

    logger.info(f"Execution Time: {end_time - start_time:.2f} seconds")

    updated_state = {"explanation": response.content}

    logger.info(f"Outgoing Partial State: {json.dumps(updated_state, indent=2)}")
    logger.info("----- EXITING NODE: generate_explanation -----\n")

    return updated_state

## Node 2: Summarize

In [0]:
def summarize(state: GraphState):
    logger.info("----- ENTERING NODE: summarize -----")
    logger.info(f"Incoming State: {json.dumps(state, indent=2)}")

    start_time = time.time()

    prompt = f"Summarize this:\n{state['explanation']}"
    logger.info(f"Prompt Sent To Model:\n{prompt}")

    response = chat_model.invoke(prompt)

    end_time = time.time()

    logger.info(f"Raw Response Object: {response}")
    logger.info(f"Response Content:\n{response.content}")

    if hasattr(response, "response_metadata"):
        logger.info(f"Token Usage Metadata: {response.response_metadata}")

    logger.info(f"Execution Time: {end_time - start_time:.2f} seconds")

    updated_state = {"summary": response.content}

    logger.info(f"Outgoing Partial State: {json.dumps(updated_state, indent=2)}")
    logger.info("----- EXITING NODE: summarize -----\n")

    return updated_state

# Step 5: Build Graph With Transition Logs

In [0]:
logger.info("Building LangGraph Workflow...")

workflow = StateGraph(GraphState)

workflow.add_node("generate", generate_explanation)
workflow.add_node("summarize", summarize)

workflow.set_entry_point("generate")
workflow.add_edge("generate", "summarize")
workflow.add_edge("summarize", END)

app = workflow.compile()

logger.info("Graph compiled successfully.")

# Step 6: Execute With Full Debug

In [0]:
logger.info("Invoking LangGraph...\n")

final_result = app.invoke({"topic": "LangGraph"})

logger.info("----- FINAL STATE -----")
logger.info(json.dumps(final_result, indent=2))

In [0]:
def clean_dict(d):
    """
    Recursively remove keys with None, empty string, or empty list/dict values.
    Flatten nested dicts for readability.
    """
    if isinstance(d, dict):
        cleaned = {}
        for k, v in d.items():
            if v is None or v == "" or v == [] or v == {}:
                continue
            if isinstance(v, dict):
                nested = clean_dict(v)
                if nested:
                    cleaned[k] = nested
            elif isinstance(v, list):
                cleaned_list = [clean_dict(item) for item in v if item not in [None, "", [], {}]]
                if cleaned_list:
                    cleaned[k] = cleaned_list
            else:
                cleaned[k] = v
        return cleaned
    else:
        return d

cleaned_result = clean_dict(final_result)
logger.info("----- CLEANED FINAL RESULT -----")
logger.info(json.dumps(cleaned_result, indent=2))
display(cleaned_result)

# End of Notebook